# 02 — Target Definition, Features, and the Leakage Proof

The most important notebook in the repository, because leakage is the failure
mode that makes a battery-RUL model look excellent and be worthless.

As before: no logic here, only calls into `src/battery_rul/`.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np, pandas as pd
from battery_rul.config import load_config
from battery_rul.data import load_cycles
from battery_rul.features import (attach_target, build_features, feature_columns,
                                  assert_no_leakage, make_split, FeaturePipeline)
from battery_rul.features.target import find_eol_cycle
from battery_rul.visualization import apply_style
from battery_rul.visualization.style import figure, battery_palette

cfg = load_config('../configs/default.yaml'); apply_style(cfg.viz)
dataset = load_cycles(cfg)

## 1. The target

```
RUL(k) = k_EOL - k
```

`k_EOL` = first **persistent** cycle at which the trailing-median capacity
falls to or below 70 % of rated. Persistence matters: cells recover capacity
after rest, so a single dip below threshold is routine.

In [ ]:
for c in sorted(dataset.frame.battery_id.unique()):
    g = dataset.battery(c)
    print(f'{c}: {len(g):4d} cycles, EOL at {find_eol_cycle(g, cfg)}')

In [ ]:
labelled, target_report = attach_target(dataset.frame, cfg)
target_report.to_dict()

### Why persistence is not cosmetic

Compare the first *bare* crossing with the first *persistent* one. Where they
differ, the bare rule would have declared the cell dead while it still had
usable life — and every RUL label for that cell would be wrong.

In [ ]:
rows = []
for c in sorted(dataset.frame.battery_id.unique()):
    g = dataset.battery(c)
    below = g.capacity_smooth_ah.to_numpy() <= cfg.eol_capacity_ah
    bare = int(g.cycle_index.to_numpy()[below][0]) if below.any() else None
    rows.append({'battery_id': c, 'first_crossing': bare,
                 'persistent_crossing': find_eol_cycle(g, cfg)})
pd.DataFrame(rows)

In [ ]:
with figure(figsize=(11, 5), cfg=cfg.viz) as (fig, ax):
    colours = battery_palette(sorted(labelled.battery_id.unique()))
    for c, g in labelled.groupby('battery_id'):
        ax.plot(g.cycle_index, g[cfg.target.name], color=colours[c], label=c)
    ax.set_xlabel('Discharge cycle'); ax.set_ylabel('RUL (cycles)')
    ax.set_title('RUL trajectory per cell'); ax.legend(ncol=4)

## 2. Feature engineering

~700 causal features from 14 base signals, pruned to ~400.

In [ ]:
features, report = build_features(labelled, cfg.features)
print(f'generated {report.n_generated}, kept {report.n_after_pruning}')
print(f'warm-up rows dropped: {report.warmup_rows_dropped}')
features.shape

In [ ]:
# A sample of what gets generated, by family
names = feature_columns(features)
for kind in ['rmean', 'rstd', 'lag', 'diff', 'slope', 'ewm', 'cum', 'ratio_to_initial']:
    matches = [n for n in names if kind in n]
    print(f'{kind:18s} {len(matches):4d}  e.g. {matches[:2]}')

## 3. The leakage proof

The claim is that a feature at cycle *k* reads only cycles ≤ *k*. The test is
mechanical: build features on the full history, then rebuild on a truncated
prefix. If any feature peeked forward, deleting the future would change it.

This runs on every pipeline execution, not just here.

In [ ]:
for c in sorted(labelled.battery_id.unique()):
    assert_no_leakage(labelled, cfg.features, battery_id=c)
print('All cells pass the causality contract.')

### The guard can actually fail

A checker that has never been observed to fail proves nothing. Here we hand it
a deliberately non-causal feature — a *reverse* cumulative minimum, which at
cycle k reads every cycle after k — and confirm it is caught.

In [ ]:
def leaky_builder(df, feature_cfg):
    frame, rep = build_features(df, feature_cfg)
    src = feature_columns(frame)[0]
    frame['planted_future_min'] = (
        frame.iloc[::-1].groupby('battery_id')[src].cummin().iloc[::-1])
    return frame, rep

try:
    assert_no_leakage(labelled, cfg.features,
                      battery_id=sorted(labelled.battery_id.unique())[0],
                      builder=leaky_builder)
    print('NOT CAUGHT — this would be a bug in the guard')
except AssertionError as exc:
    print('Caught, as required:'); print(str(exc)[:300])

## 4. Splitting

Never random. Whole cells are held out — the deployment question.

In [ ]:
split = make_split(features, cfg.split)
split.to_dict()

## 5. The fitted transform

Scaling and supervised top-K selection are fitted on **training rows only**.
This object is what ships as `models/feature_pipeline.pkl` and is what serving
loads — which is how training/serving skew is prevented.

In [ ]:
names = feature_columns(features)
y = features[cfg.target.name].to_numpy()
pipe = FeaturePipeline(cfg=cfg.features).fit(features.loc[split.train, names], y[split.train])
print(pipe)
pd.Series(pipe.selection_scores).sort_values(ascending=False).head(15)

---
**Next:** `03_model_comparison.ipynb`.